#### Setting up the network and data

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import numpy as np
from tqdm.auto import tqdm
import h5py
import plotly.graph_objects as go
from torch.func import functional_call
import matplotlib.pyplot as plt

from loss_landscapes.paths import PROJ_ROOT
from loss_landscapes.utils import plotly_template
plotly_template.use("mine_dark")


# the network
class Block(nn.Module):
    def __init__(self, ch: int, skip: bool):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels=ch, out_channels=ch, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(num_features=ch)
        self.conv2 = nn.Conv2d(in_channels=ch, out_channels=ch, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(num_features=ch)
        self.skip = skip
        self.activ = nn.ReLU()
        
    def forward(self, x):
        out = self.activ(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        if self.skip:
            return self.activ(out + x)
        else:
            return self.activ(out)

class SmallResNet(nn.Module):
    def __init__(self, ch: int = 16, blocks_per_stage: int = 9, skip: bool = True, n_classes: int = 10):
        super().__init__()
        
        # this goes from greyscale img to ch "hidden" channels that are used in blocks
        # I guess this is also a "transition layer" and could go in the stages logic
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(num_features=ch),
            nn.ReLU(),
        )
        
        # this is the main chain of blocks
        # multiple stages, each a bunch of blocks with the same params
        # as we go from one stage to the next -> Res/2 while ch*2, so a transition layer is needed
        n_stages = 3
        stages = []
        for s in range(n_stages):
            c = ch * 2**s # doubles the number of channels for each progressive stage (as the resolution is halved)
            
            if s > 0:
                stages += [
                    nn.Conv2d(c//2, c, 3, stride=2, padding=1, bias=False), 
                    nn.BatchNorm2d(c),
                    nn.ReLU()
                ]
            stages += [Block(c, skip) for _ in range(blocks_per_stage)] # create a new block! not just []*b_p_s!  
        self.stages = nn.Sequential(*stages)
        
        # the output with mlp to get to a flat n_classes output
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch * 2**(n_stages-1), n_classes)
        )

    def forward(self, x):
        return self.head(self.stages(self.stem(x)))
        
 
# Device setup 
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"⯈ using {DEVICE} as device")


# load dataset
print("⯈ loading dataset ...")
with h5py.File(PROJ_ROOT / "data/ds_fashion.h5", "r") as f:
    X = f["data"][...]
    y = f["targets"][...]


# define some vars
n_epochs = 5
batch_size = 128
train_perc = 0.8
n_samples = len(X)
n_train = int(n_samples * train_perc)
n_valid = n_samples - n_train


# normalize dataset (with statistics determined from the training part only)
mu = X[:n_train].mean() # scalar
si = X[:n_train].std() # scalar
X = (X - mu) / si # ... but apply to entire dataset!

# setup training elements
loss_fn = nn.CrossEntropyLoss(reduction="mean")
model = SmallResNet(skip=False).to(DEVICE)
optim = torch.optim.Adam(params=model.parameters(), lr=1e-3)
X = torch.Tensor(X)[:, None, :, :].to(DEVICE) # add the 1 channel singleton dim for conv layers
y = torch.Tensor(y).to(torch.long).to(DEVICE)

#### Training the network

In [ ]:
for E in tqdm(range(n_epochs), desc="training epoch: ", position=0):
    
    # training part -----
    train_idx = torch.randperm(n_train, device=DEVICE)
    model.train()
    losses_train = []
    
    for batch_start in range(0, n_train, batch_size):
        batch_idx = train_idx[batch_start:batch_start+batch_size]

        optim.zero_grad()
        pred = model(X[batch_idx])
        loss = loss_fn(pred, y[batch_idx])
        loss.backward()
        optim.step()
        
        losses_train.append(loss.detach())
      
        
    # validation part -----
    valid_idx = torch.arange(n_valid, device=DEVICE) + n_train # offset by n train to not use train samples!
    losses_valid = []
    accs_valid = []
    model.eval()
    
    with torch.no_grad():
        for batch_start in range(0, n_valid, 1_000):
            batch_idx = valid_idx[batch_start:batch_start+1_000]
            
            # valid loss
            pred = model(X[batch_idx])
            loss = loss_fn(pred, y[batch_idx])
            losses_valid.append(loss.detach())
            
            # valid accuracy
            labels = torch.argmax(pred, dim=1)
            acc = (labels == y[batch_idx]).float().mean()
            accs_valid.append(acc)
    
    # log the losses
    tqdm.write(
        f"L_train = {torch.stack(losses_train).mean().item():.4f} | "
        f"L_valid = {torch.stack(losses_valid).mean().item():.4f} | "
        f"A_valid = {torch.stack(accs_valid).mean().item():.2f}"
    )

In [ ]:
ckpt_dir = PROJ_ROOT / "checkpoints"
ckpt_dir.mkdir(exist_ok=True)

config = dict(
    ch=16, 
    blocks_per_stage=9, 
    skip=False,
)

torch.save(
    {"config": config, "state_dict": model.state_dict(), "mu": float(mu), "si": float(si)},
    ckpt_dir / "resnet_b9_noskip.pt",
)

#### Naive version again with resnet

In [ ]:
def loss_at(params: dict, X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """ to evaluate a model with data AND parameters """
    pred = functional_call(model, params, (X, ))
    loss = loss_fn(pred, y)

    return loss

# two directions in parameter space + the trained weights fixed point
theta = {k: v.detach().clone() for k, v in model.named_parameters()}
d1 = {k: torch.randn_like(v) for k, v in model.named_parameters()}
d2 = {k: torch.randn_like(v) for k, v in model.named_parameters()}


reach = 0.5
N = 20
alphas = np.linspace(-reach, reach, N)
betas  = np.linspace(-reach, reach, N)

evalX = X[:100]
evaly = y[:100]
Z = torch.zeros(len(betas), len(alphas))

with torch.no_grad():
    for j, b in tqdm(enumerate(betas)):
        for i, a in enumerate(alphas):
            params = {k: theta[k] + a*d1[k] + b*d2[k] for k in theta}
            Z[i, j] = loss_at(params, evalX, evaly)

    
fig = go.Figure()
fig.add_trace(
    go.Surface(
        x = alphas,
        y = betas,
        z = np.log(Z.numpy()),
    )
)

fig.show()

#### generating loss landscape with filter nomalization on the random directions

In [ ]:
def loss_at(params: dict, X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """ to evaluate a model with data AND parameters """
    pred = functional_call(model, params, (X, ))
    loss = loss_fn(pred, y)
    return loss

def filternorm(tens: torch.Tensor):
    """ one norm per output filter/neuron: norm over all dims except dim 0 """
    norms = tens.flatten(1).norm(dim=1) # shape (out,)
    return norms.view(-1, *[1] * (tens.dim() - 1)) # (out, 1, 1, 1) or (out, 1), so it broadcasts
    
def generate_dir(param_snapshot) -> dict:
    dirdict = {}
    
    for k, p in param_snapshot.items():
        
        if p.dim() > 1:
        
            randn_dir = torch.randn_like(p)
            normed_dir = randn_dir / filternorm(randn_dir)
            scaled_dir = normed_dir * filternorm(p)
            dirdict[k] = scaled_dir
        
        else: 
            # biases and batch norm should stay at their trained values
            dirdict[k] = torch.zeros_like(p)

    return dirdict
  
ckpt = torch.load(PROJ_ROOT / "checkpoints/resnet_b9_noskip.pt", map_location=DEVICE)
model = SmallResNet(**ckpt["config"]).to(DEVICE)
model.load_state_dict(ckpt["state_dict"])
            
# two directions in parameter space + the trained weights fixed point
theta = {k: v.detach().clone() for k, v in model.named_parameters()}
d1 = generate_dir(theta)
d2 = generate_dir(theta)

reach = 0.8
N = 81

alphas = np.linspace(-reach, reach, N)
betas  = np.linspace(-reach, reach, N)

evalX = X[:100]
evaly = y[:100]
Z = torch.zeros(len(betas), len(alphas))

model.eval() # so that running batch norm statistics are kept constant
with torch.no_grad():
    for j, b in tqdm(enumerate(betas), total=N):
        for i, a in enumerate(alphas):
            params = {k: theta[k] + a*d1[k] + b*d2[k] for k in theta}
            Z[j, i] = loss_at(params, evalX, evaly)
            
fig = go.Figure()
fig.add_trace(
    go.Surface(
        x = alphas,
        y = betas,
        z = np.log(Z.numpy()),
    )
)

fig.show(renderer="browser")

In [ ]:
fig = go.Figure(
    data = go.Surface(
        x = alphas,
        y = betas,
        z = np.log(np.clip(Z.numpy(), a_min=0, a_max=10)),
        colorscale="dense",
    ),
    layout = dict(
        margin=dict(l=20, r=20, b=20, t=20),
    ),
)

fig.show()